# Lab 5 - Consumer and MITRE Classifier

In this notebook I read the raw events from Kafka, classify them with simple MITRE ATT&CK rules, and save the result to `classified_packets.csv`.

## What This Notebook Does

- Reads events from the `raw-events` Kafka topic.
- Uses simple rule-based MITRE mapping.
- Saves the classified records to a CSV file.
- Adds tracing spans for consuming, classifying, and writing.

In [1]:
%pip install -q kafka-python opentelemetry-sdk opentelemetry-exporter-otlp

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import json

from kafka import KafkaConsumer

from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

KAFKA_BOOTSTRAP_SERVERS = "kafka:9092"
KAFKA_TOPIC = "raw-events"
KAFKA_GROUP_ID = "lab5-classifier"
OTLP_ENDPOINT = "jaeger:4317"
SERVICE_NAME = "lab5-classifier"
OUTPUT_CSV = "classified_packets.csv"

resource = Resource.create({"service.name": SERVICE_NAME})
trace.set_tracer_provider(TracerProvider(resource=resource))
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(OTLPSpanExporter(endpoint=OTLP_ENDPOINT, insecure=True))
)
tracer = trace.get_tracer(__name__)

consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    group_id=KAFKA_GROUP_ID,
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    consumer_timeout_ms=10000,
    value_deserializer=lambda value: json.loads(value.decode("utf-8")),
)

print("Consumer connected to Kafka")

Consumer connected to Kafka


In [3]:
def classify_event(event):
    if event["event_type"] == "process_start":
        command = event.get("command_line", "").lower()

        if "encodedcommand" in command:
            return "TA0002", "T1059.001", "PowerShell encoded command"
        if "cmd.exe" in command:
            return "TA0002", "T1059.003", "Windows command shell"
        return "TA0002", "T1059", "Command execution"

    if event["event_type"] == "user_login":
        if event.get("logon_type") == "failure":
            return "TA0006", "T1110", "Possible brute force"
        return "TA0001", "T1078", "Valid account login"

    return "TA0000", "T0000", "Unknown event"

In [4]:
fieldnames = [
    "event_id",
    "timestamp",
    "user",
    "host",
    "source_ip",
    "event_type",
    "mitre_tactic",
    "mitre_technique",
    "classification_reason",
]

csv_file = open(OUTPUT_CSV, "w", newline="", encoding="utf-8")
writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
writer.writeheader()

print(f"Writing results to {OUTPUT_CSV}")

Writing results to classified_packets.csv


In [5]:
MAX_EVENTS = 30
processed = 0

for message in consumer:
    event = message.value

    with tracer.start_as_current_span("consume_event") as span:
        span.set_attribute("event.id", event["event_id"])
        span.set_attribute("event.type", event["event_type"])

        with tracer.start_as_current_span("classify_event"):
            tactic, technique, reason = classify_event(event)

        record = {
            "event_id": event["event_id"],
            "timestamp": event["timestamp"],
            "user": event["user"],
            "host": event["host"],
            "source_ip": event["source_ip"],
            "event_type": event["event_type"],
            "mitre_tactic": tactic,
            "mitre_technique": technique,
            "classification_reason": reason,
        }

        with tracer.start_as_current_span("write_csv"):
            writer.writerow(record)
            csv_file.flush()

    processed += 1
    print(f"Processed {processed}: {tactic} / {technique} - {reason}")

    if processed >= MAX_EVENTS:
        break

csv_file.close()
consumer.close()
print(f"Finished processing {processed} events")

Processed 1: TA0006 / T1110 - Possible brute force
Processed 2: TA0006 / T1110 - Possible brute force
Processed 3: TA0001 / T1078 - Valid account login
Processed 4: TA0001 / T1078 - Valid account login
Processed 5: TA0001 / T1078 - Valid account login
Processed 6: TA0002 / T1059 - Command execution
Processed 7: TA0001 / T1078 - Valid account login
Processed 8: TA0001 / T1078 - Valid account login
Processed 9: TA0002 / T1059.001 - PowerShell encoded command
Processed 10: TA0001 / T1078 - Valid account login
Processed 11: TA0006 / T1110 - Possible brute force
Processed 12: TA0002 / T1059.003 - Windows command shell
Processed 13: TA0006 / T1110 - Possible brute force
Processed 14: TA0006 / T1110 - Possible brute force
Processed 15: TA0002 / T1059.003 - Windows command shell
Processed 16: TA0006 / T1110 - Possible brute force
Processed 17: TA0002 / T1059 - Command execution
Processed 18: TA0001 / T1078 - Valid account login
Processed 19: TA0001 / T1078 - Valid account login
Processed 20: T